# 01 -- Data Preparation & EDA

**Purpose:** Understand the raw Vehicle Insurance Customer dataset — shape, distributions, correlations, and missing values.

| Step | Description |
|---|---|
| 1 | Data Overview |
| 2 | Missing Value Analysis |
| 3 | Univariate Analysis |
| 4 | Bivariate & Correlation Analysis |
| 5 | Key Findings |

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_PATH, IMAGES_DIR, NUMERICAL_COLS, TARGET_COL
from src.data_loader import load_raw_data, validate_data

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Data Overview

In [2]:
df = load_raw_data(RAW_DATA_PATH)
validate_data(df)

Loaded 9,134 rows x 24 cols from AutoInsurance.csv

-- Shape: (9134, 24)
-- Dtypes:
 object     16
int64       6
float64     2
Name: count, dtype: int64
-- No missing values
-- Duplicate rows: 0


In [3]:
df.head()

,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,Income,Location Code,Marital Status,Monthly Premium Auto,Months Since Last Claim,Months Since Policy Inception,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size
0,BU79786,Washington,2763.52,No,Basic,Bachelor,2/24/11,Employed,F,56274,Suburban,Married,69,32,5,0,1,Corporate Auto,Corporate L3,Offer1,Agent,384.81,Two-Door Car,Medsize
1,QZ44356,Arizona,6979.54,No,Extended,Bachelor,1/31/11,Unemployed,F,0,Suburban,Single,94,13,42,0,8,Personal Auto,Personal L3,Offer3,Agent,1131.46,Four-Door Car,Medsize
2,AI49188,Nevada,12887.43,No,Premium,Bachelor,2/19/11,Employed,F,48767,Suburban,Married,108,18,38,0,2,Personal Auto,Personal L3,Offer1,Agent,566.47,Two-Door Car,Medsize
3,WW63253,California,7645.86,No,Basic,Bachelor,1/20/11,Unemployed,M,0,Suburban,Married,106,18,65,0,7,Corporate Auto,Corporate L2,Offer1,Call Center,529.88,SUV,Medsize
4,HB64268,Washington,2813.69,No,Basic,Bachelor,3/2/2011,Employed,M,43836,Rural,Single,73,12,44,0,1,Personal Auto,Personal L1,Offer1,Agent,138.13,Four-Door Car,Medsize


In [4]:
df.describe()

,Customer Lifetime Value,Income,Monthly Premium Auto,Months Since Last Claim,Months Since Policy Inception,Number of Open Complaints,Number of Policies,Total Claim Amount
count,9134.00,9134.00,9134.00,9134.00,9134.00,9134.00,9134.00,9134.00
mean,8004.94,37657.38,93.22,15.10,48.06,0.38,2.97,434.09
std,6870.97,30379.90,34.41,10.07,27.91,0.91,2.39,290.50
min,1898.01,0.00,61.00,0.00,0.00,0.00,1.00,0.10
25%,3994.25,0.00,68.00,6.00,24.00,0.00,1.00,272.26
50%,5780.18,33889.50,83.00,14.00,48.00,0.00,2.00,383.95
75%,8962.17,62320.00,109.00,23.00,71.00,0.00,4.00,547.51
max,83325.38,99981.00,298.00,35.00,99.00,5.00,9.00,2893.24


In [5]:
print("Duplicates:", df.duplicated().sum())
print("\nValue counts -- Response:")
print(df['Response'].value_counts())

Duplicates: 0

Value counts -- Response:
Response
No     7826
Yes    1308
Name: count, dtype: int64


## 2. Missing Value Analysis

In [6]:
missing = df.isnull().sum().sort_values(ascending=False)
print("Missing values per column:")
print(missing)
print("\nNo missing values!" if missing.sum() == 0 else f"Total missing cells: {missing.sum()}")

fig, ax = plt.subplots(figsize=(10, 4))
if missing.sum() == 0:
    ax.text(0.5, 0.5, 'No Missing Values', transform=ax.transAxes,
            ha='center', va='center', fontsize=18, color='green')
else:
    missing[missing > 0].plot(kind='bar', ax=ax, color='coral')
    ax.set_title('Missing Values per Feature')
ax.set_title('Missing Value Analysis')
fig.savefig(IMAGES_DIR / 'missing_values.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: missing_values.png")

Missing values per column:
Customer                         0
State                            0
Customer Lifetime Value          0
Response                         0
Coverage                         0
Education                        0
Effective To Date                0
EmploymentStatus                 0
Gender                           0
Income                           0
Location Code                    0
Marital Status                   0
Monthly Premium Auto             0
Months Since Last Claim          0
Months Since Policy Inception    0
Number of Open Complaints        0
Number of Policies               0
Policy Type                      0
Policy                           0
Renew Offer Type                 0
Sales Channel                    0
Total Claim Amount               0
Vehicle Class                    0
Vehicle Size                     0
dtype: int64

No missing values!


Saved: missing_values.png


## 3. Univariate Analysis

In [7]:
# Target distribution -- raw vs log
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[TARGET_COL], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Customer Lifetime Value (raw)')
axes[0].set_xlabel('CLV ($)')

sns.histplot(np.log1p(df[TARGET_COL]), kde=True, ax=axes[1], color='seagreen')
axes[1].set_title('log1p(Customer Lifetime Value)')
axes[1].set_xlabel('log(CLV)')

fig.suptitle('Target Variable Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'target_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: target_distribution.png")
print(f"\nCLV Skewness (raw):    {df[TARGET_COL].skew():.4f}")
print(f"CLV Skewness (log):    {np.log1p(df[TARGET_COL]).skew():.4f}")

Saved: target_distribution.png

CLV Skewness (raw):    3.0323
CLV Skewness (log):    0.5762


In [8]:
# Numerical feature distributions
num_cols = [c for c in NUMERICAL_COLS if c in df.columns]
n = len(num_cols)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='mediumpurple')
    axes[i].set_title(col)
    axes[i].set_xlabel('')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: numerical_distributions.png")

Saved: numerical_distributions.png


In [9]:
# Categorical feature distributions
cat_cols = ['State', 'Response', 'Coverage', 'Education', 'EmploymentStatus',
            'Gender', 'Location Code', 'Marital Status', 'Vehicle Class', 'Vehicle Size']
cat_cols = [c for c in cat_cols if c in df.columns]

n = len(cat_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    vc = df[col].value_counts()
    vc.plot(kind='bar', ax=axes[i], color='cornflowerblue')
    axes[i].set_title(col)
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=35, ha='right')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: categorical_distributions.png")

Saved: categorical_distributions.png


## 4. Bivariate & Correlation Analysis

In [10]:
# CLV by Coverage
fig, ax = plt.subplots(figsize=(10, 5))
order = df.groupby('Coverage')[TARGET_COL].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Coverage', y=TARGET_COL, order=order, ax=ax, palette='Set2')
ax.set_title('CLV by Coverage Type')
ax.set_ylabel('Customer Lifetime Value ($)')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'clv_by_coverage.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: clv_by_coverage.png")

Saved: clv_by_coverage.png


In [11]:
# CLV by Vehicle Class
fig, ax = plt.subplots(figsize=(12, 5))
order = df.groupby('Vehicle Class')[TARGET_COL].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Vehicle Class', y=TARGET_COL, order=order, ax=ax, palette='Set3')
ax.set_title('CLV by Vehicle Class')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'clv_by_vehicle_class.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: clv_by_vehicle_class.png")

Saved: clv_by_vehicle_class.png


In [12]:
# Correlation heatmap
num_cols_target = num_cols + [TARGET_COL]
fig, ax = plt.subplots(figsize=(12, 8))
corr = df[num_cols_target].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 10})
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: correlation_heatmap.png")

# Print top correlations with target
target_corr = corr[TARGET_COL].drop(TARGET_COL).sort_values(key=abs, ascending=False)
print("\nTop correlations with CLV:")
print(target_corr)

Saved: correlation_heatmap.png

Top correlations with CLV:
Monthly Premium Auto             0.40
Total Claim Amount               0.23
Number of Open Complaints       -0.04
Income                           0.02
Number of Policies               0.02
Months Since Last Claim          0.01
Months Since Policy Inception    0.01
Name: Customer Lifetime Value, dtype: float64


In [13]:
# Scatter: Monthly Premium vs CLV
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df['Monthly Premium Auto'], df[TARGET_COL], alpha=0.3, s=10, color='steelblue')
ax.set_xlabel('Monthly Premium Auto ($)')
ax.set_ylabel('Customer Lifetime Value ($)')
ax.set_title('Monthly Premium vs CLV')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'premium_vs_clv.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: premium_vs_clv.png")

Saved: premium_vs_clv.png


## 5. Key Findings

| Finding | Detail |
|---|---|
| Dataset size | 9,134 rows, 24 columns |
| Missing values | None |
| Duplicates | 0 |
| Target (CLV) skewness | High right skew -- log transform required |
| Strongest predictor | Monthly Premium Auto (highest positive correlation with CLV) |
| Coverage impact | Extended coverage customers have highest median CLV |
| Vehicle class | Luxury Car and Sports Car customers show highest CLV |
| Employment | Employed customers show higher CLV than unemployed |
| Target range | $1,898 -- $83,325 (mean $8,005) |

---
**Next step: `02_Feature_Engineering.ipynb`**